In [1]:
PROJECT_RANDOM_STATE = 2627
print("PROJECT_RANDOM_STATE =", PROJECT_RANDOM_STATE)

PROJECT_RANDOM_STATE = 2627


# Practice 4 — Data and representation for trees (R1)

> **Team code style (delete this cell before submitting).** The team must be able to defend every line.
>
> - Prefer tools seen in class or named in the statement. If you cannot explain a line in one sentence, rewrite it.
> - Plain `for` loops and `if`s, one step per line, clear variable names. No lambdas, classes or long chains.
> - Same results, simpler code. After simplifying, restart, run all and check that the numbers did not change.
> - Interpretation goes in Markdown cells, in English. Check your notebook with `revisor_notebook.py`.
> - Every number written in a Markdown cell must appear in a printed output of the notebook.
>
> **Shared names, so that the notebooks can be joined:** `dev` (development table), `X`, `y` (cyber target), `z` (storage target), `latency_ns`, `NOMINAL`, `NUMERIC`, `FEATURES`, `make_preprocessor()`, `folds` (list of `(train_idx, val_idx)`, also passed as `cv=folds`), `CLASS_NAMES` (cyber).

## Setup

In [2]:
import json
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Paths are relative to the notebooks/ folder, where this file is run.
CYBER_DIR = "../datos/cyber"
STORAGE_DIR = "../datos/storage"
FOLDS_DIR = "../folds"

# Challenge A — cyber

## Data

In [3]:
# Load the tables and the class codes without modifying the original files.
dev = pd.read_csv(CYBER_DIR + "/cyber_development.csv")
cyber_groups = pd.read_csv(CYBER_DIR + "/cyber_groups.csv")
cyber_folds = pd.read_csv(FOLDS_DIR + "/cyber_folds.csv")

# The JSON keys are text; convert them to the integer codes used in y.
with open(CYBER_DIR + "/cyber_class_codes.json", encoding="utf-8") as file:
    class_codes = json.load(file)

CLASS_NAMES = {}
for code, name in class_codes.items():
    CLASS_NAMES[int(code)] = name

# The 11 source predictors of P2. Metadata columns are never predictors.
NOMINAL = ["proto", "service", "conn_state"]
NUMERIC = ["duration", "src_bytes", "dst_bytes", "missed_bytes",
           "src_pkts", "src_ip_bytes", "dst_pkts", "dst_ip_bytes"]
FEATURES = NOMINAL + NUMERIC

expected_columns = ["row_id"] + FEATURES + ["attack_code"]
assert list(dev.columns) == expected_columns, "Unexpected cyber schema"

X = dev[FEATURES].copy()
y = dev["attack_code"].copy()
for column in ["row_id", "feature_group", "config_group", "fold_id", "attack_code"]:
    assert column not in X.columns, "Forbidden predictor: " + column

assert X.shape[1] == 11, "Expected 11 source predictors"
assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert pd.api.types.is_integer_dtype(y), "Class codes must be integers"
assert sorted(y.unique()) == sorted(CLASS_NAMES), "Class codes do not match JSON"

print("Development table:", dev.shape)
print("Predictors:", X.shape, "(", len(NOMINAL), "nominal and", len(NUMERIC), "numeric )")
print("Class names:", CLASS_NAMES)

Development table: (34928, 13)
Predictors: (34928, 11) ( 3 nominal and 8 numeric )
Class names: {0: 'backdoor', 1: 'ddos', 2: 'dos', 3: 'injection', 4: 'mitm', 5: 'normal', 6: 'password', 7: 'ransomware', 8: 'scanning', 9: 'xss'}


In [4]:
# Check that the three tables are aligned before using positions.
assert list(cyber_groups.columns) == ["row_id", "feature_group"], "Unexpected group schema"
assert list(cyber_folds.columns) == ["row_id", "fold_id"], "Unexpected fold schema"
assert dev["row_id"].is_unique, "Duplicate development row_id"
assert dev["row_id"].equals(cyber_groups["row_id"]), "Group rows are not aligned"
assert dev["row_id"].equals(cyber_folds["row_id"]), "Fold rows are not aligned"
assert sorted(cyber_folds["fold_id"].unique()) == [0, 1, 2, 3, 4], "Expected five folds"

# Read the official assignment: the folds are never rebuilt.
folds = []
fold_rows = []
for fold_id in range(5):
    train_idx = np.flatnonzero(cyber_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(cyber_folds["fold_id"].to_numpy() == fold_id)

    # A group never appears on both sides of the same split.
    train_groups = set(cyber_groups.iloc[train_idx]["feature_group"])
    val_groups = set(cyber_groups.iloc[val_idx]["feature_group"])
    assert train_groups.isdisjoint(val_groups), "Group leakage in fold " + str(fold_id)
    assert len(train_idx) + len(val_idx) == len(dev), "Incomplete partition"

    folds.append((train_idx, val_idx))
    fold_rows.append({"fold_id": fold_id, "training_rows": len(train_idx),
                      "validation_rows": len(val_idx)})

print("Feature groups:", cyber_groups["feature_group"].nunique())
print(pd.DataFrame(fold_rows).to_string(index=False))
print("All five canonical folds passed the group separation checks.")

Feature groups: 34883
 fold_id  training_rows  validation_rows
       0          27942             6986
       1          27942             6986
       2          27943             6985
       3          27942             6986
       4          27943             6985
All five canonical folds passed the group separation checks.


The cyber development table has 34,928 rows and 11 source predictors (3 nominal and 8 numeric). The target is `attack_code`, with the 10 class names supplied in the JSON file. There are 34,883 feature groups. The five canonical folds are read from `../folds/cyber_folds.csv`, never rebuilt, and the checks above confirm that no group appears on both sides of a split.

Identifiers, grouping metadata, fold assignments and the target are kept outside `X`. Nothing has been fitted in this block.

## Representation decisions (statement, Section 2)

**Scaling.** The tree gets no scaler and no `log1p`. A tree asks questions of the form "is `x <= t`?". Any strictly increasing transformation of a numeric column, such as standardisation or `log1p`, changes the threshold `t` but not which rows fall on each side of it, so scaling adds nothing for a tree. Dropping `log1p` also keeps every threshold in the original units (bytes, packets, seconds), so R2 can read a threshold on `dst_bytes` directly in bytes, without converting it back with `e^x - 1`. The eight numeric predictors enter the model unchanged (`"passthrough"`).

**Categorical encoding.** `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` with no level removed. P3 removed one reference level per predictor because the linear models have an intercept. A tree has no intercept and needs no reference level, and removing one would hide that level: with `tcp` removed, the tree could not ask "is the protocol tcp?".

**Unseen levels.** With `handle_unknown="ignore"`, a level that training never saw is encoded as zeros for its predictor. In P3 this coincided with the encoding of the dropped reference level. In P4 no level is dropped, so all zeros now means "none of the known levels". This happens in the data: `service=gssapi` is absent from the training rows of fold 3.

### Correspondence with P3

In [5]:
# Levels of each nominal predictor in the development table, and the P3 mapping.
# P3 dropped one reference level per predictor (the most frequent one).
P3_DROPPED_LEVELS = ["tcp", "-", "SF"]

mapping_rows = []
for column, dropped in zip(NOMINAL, P3_DROPPED_LEVELS):
    levels = sorted(X[column].unique())
    assert dropped in levels, "P3 reference level not found: " + dropped
    mapping_rows.append({"predictor": column, "levels": len(levels),
                         "P3_dropped_level": dropped,
                         "P3_indicator_columns": len(levels) - 1,
                         "P4_indicator_columns": len(levels)})

mapping = pd.DataFrame(mapping_rows)
print(mapping.to_string(index=False))
print("Total indicator columns, P3:", mapping["P3_indicator_columns"].sum())
print("Total indicator columns, P4:", mapping["P4_indicator_columns"].sum())

 predictor  levels P3_dropped_level  P3_indicator_columns  P4_indicator_columns
     proto       3              tcp                     2                     3
   service       9                -                     8                     9
conn_state      13               SF                    12                    13
Total indicator columns, P3: 22
Total indicator columns, P4: 25


P3 dropped `proto=tcp`, `service=-` and `conn_state=SF`, and it standardised the log-transformed numeric predictors. P4 keeps those three levels as the columns `proto_tcp`, `service_-` and `conn_state_SF`, and applies neither `log1p` nor a scaler. The three predictors have 3, 9 and 13 levels, so the indicator columns go from 22 in P3 to 25 in P4. The check below confirms, on every fold, that each P3 indicator column has exactly the same values as the P4 column of the same level; in fold 3 the counts are 21 and 24 because `gssapi` is missing from its training vocabulary.

## Preprocessing block (publish for R2 and R4)

In [6]:
def make_preprocessor():
    # One-hot with every level kept: a tree needs no reference level.
    # "ignore" encodes a level unseen in training as all zeros.
    nominal_steps = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    # Numeric predictors pass through unchanged: no log1p and no scaler.
    preprocessor = ColumnTransformer([
        ("cat", nominal_steps, NOMINAL),
        ("num", "passthrough", NUMERIC),
    ])
    # Returned unfitted so that R2 puts it inside its Pipeline.
    return preprocessor

In [7]:
# Check the block inside each canonical fold: fitted on training rows only.
check_rows = []
for fold_id, (train_idx, val_idx) in enumerate(folds):
    preprocessor = make_preprocessor()
    assert not hasattr(preprocessor, "transformers_"), "Expected an unfitted transformer"
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]
    X_train_t = preprocessor.fit_transform(X_train)
    X_valid_t = preprocessor.transform(X_valid)

    encoder = preprocessor.named_transformers_["cat"]
    assert encoder.drop_idx_ is None, "No level may be dropped"

    # Count the columns and the validation levels that training never saw.
    n_cat = 0
    unseen_per_row = np.zeros(len(X_valid), dtype=int)
    for column, categories in zip(NOMINAL, encoder.categories_):
        assert list(categories) == sorted(X_train[column].unique()), "Wrong vocabulary"
        n_cat = n_cat + len(categories)
        unseen_per_row = unseen_per_row + (~X_valid[column].isin(categories)).to_numpy()
    n_num = len(NUMERIC)

    feature_names = preprocessor.get_feature_names_out()
    assert X_train_t.shape[1] == len(feature_names), "Missing feature names"
    assert X_valid_t.shape[1] == len(feature_names), "Inconsistent columns"
    assert len(feature_names) == n_cat + n_num, "Unexpected number of columns"
    assert np.isfinite(X_train_t).all(), "Non-finite training values"
    assert np.isfinite(X_valid_t).all(), "Non-finite validation values"

    # Numeric columns keep their original values: nothing was scaled.
    assert np.array_equal(X_train_t[:, n_cat:], X_train[NUMERIC].to_numpy()), "Train values changed"
    assert np.array_equal(X_valid_t[:, n_cat:], X_valid[NUMERIC].to_numpy()), "Valid values changed"

    # One indicator per predictor is 1 in training; an unseen level gives 0.
    train_ones = X_train_t[:, :n_cat].sum(axis=1)
    valid_ones = X_valid_t[:, :n_cat].sum(axis=1)
    assert (train_ones == len(NOMINAL)).all(), "Each training row needs one level per predictor"
    assert (valid_ones == len(NOMINAL) - unseen_per_row).all(), "Unseen levels must give zeros"

    if fold_id == 0:
        names_fold_0 = list(feature_names)

    check_rows.append({"fold_id": fold_id, "output_features": len(feature_names),
                       "indicator_columns": n_cat, "numeric_columns": n_num,
                       "unseen_validation_rows": int((unseen_per_row > 0).sum())})

print(pd.DataFrame(check_rows).to_string(index=False))
print("Fold 0 output names:", names_fold_0)
print("Preprocessing checks passed on all five folds.")

 fold_id  output_features  indicator_columns  numeric_columns  unseen_validation_rows
       0               33                 25                8                       0
       1               33                 25                8                       0
       2               33                 25                8                       0
       3               32                 24                8                       1
       4               33                 25                8                       0
Fold 0 output names: ['cat__proto_icmp', 'cat__proto_tcp', 'cat__proto_udp', 'cat__service_-', 'cat__service_dce_rpc', 'cat__service_dns', 'cat__service_ftp', 'cat__service_gssapi', 'cat__service_http', 'cat__service_smb', 'cat__service_smb;gssapi', 'cat__service_ssl', 'cat__conn_state_OTH', 'cat__conn_state_REJ', 'cat__conn_state_RSTO', 'cat__conn_state_RSTOS0', 'cat__conn_state_RSTR', 'cat__conn_state_RSTRH', 'cat__conn_state_S0', 'cat__conn_state_S1', 'cat__conn_state_S2', 'ca

In [8]:
# Every P3 indicator column must equal the P4 column of the same level.
for fold_id, (train_idx, val_idx) in enumerate(folds):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    p4 = make_preprocessor()
    p4.fit(X_train)
    p4_names = list(p4.named_transformers_["cat"].get_feature_names_out(NOMINAL))
    p4_valid = p4.named_transformers_["cat"].transform(X_valid[NOMINAL])

    p3 = OneHotEncoder(drop=P3_DROPPED_LEVELS, handle_unknown="ignore", sparse_output=False)
    p3.fit(X_train[NOMINAL])
    p3_names = list(p3.get_feature_names_out(NOMINAL))
    p3_valid = p3.transform(X_valid[NOMINAL])

    for position, name in enumerate(p3_names):
        assert name in p4_names, "P3 column missing in P4: " + name
        same_column = p4_valid[:, p4_names.index(name)]
        assert np.array_equal(p3_valid[:, position], same_column), "Different values: " + name
    print("Fold", fold_id, ": P3 columns", len(p3_names), "are all in P4 columns", len(p4_names))

Fold 0 : P3 columns 22 are all in P4 columns 25
Fold 1 : P3 columns 22 are all in P4 columns 25
Fold 2 : P3 columns 22 are all in P4 columns 25


/home/fernando/anaconda3/lib/python3.14/site-packages/sklearn/preprocessing/_encoders.py:262: UserWarning: Found unknown categories in columns [1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


Fold 3 : P3 columns 21 are all in P4 columns 24
Fold 4 : P3 columns 22 are all in P4 columns 25


In [9]:
# The real unseen level of the statement: service=gssapi in fold 3.
train_idx, val_idx = folds[3]
preprocessor = make_preprocessor()
preprocessor.fit(X.iloc[train_idx])
encoder = preprocessor.named_transformers_["cat"]
assert "gssapi" not in encoder.categories_[1], "gssapi should be unseen in fold 3"

validation_rows = X.iloc[val_idx]
unseen_row = validation_rows.loc[validation_rows["service"] == "gssapi", NOMINAL]
print("Validation rows with service=gssapi in fold 3:", len(unseen_row))

# Positions of the service block inside the encoded row.
start = len(encoder.categories_[0])
stop = start + len(encoder.categories_[1])
encoded = encoder.transform(unseen_row)
print("Service indicators set to 1 for that row:", int(encoded[0, start:stop].sum()))
assert encoded[:, start:stop].sum() == 0, "An unseen service must be all zeros"

Validation rows with service=gssapi in fold 3: 1
Service indicators set to 1 for that row: 0


**Checks.** All checks pass on the five canonical folds. The transformed matrix has 33 columns in folds 0, 1, 2 and 4 (25 indicators and 8 numeric) and 32 in fold 3 (24 indicators and 8 numeric), because the training rows of fold 3 contain no `service=gssapi`. Within each fold, training and validation have the same columns, the numeric columns keep their original values, and every training row has exactly one indicator set per nominal predictor.

The only validation row with an unseen level is the `service=gssapi` row of fold 3. Its service block is all zeros, as the last check shows. scikit-learn prints an unknown-category warning the first time it meets that row: this is expected, because the encoder applies the declared all-zero fallback. These are preprocessing checks, not model results.

`make_preprocessor()` returns a new, unfitted transformer. R2 must place it inside the model `Pipeline`, so that the encoder vocabulary is learned only from each fold's training rows. The output names follow the pattern shown above (`cat__service_http`, `num__dst_bytes`), and they differ between folds, so importances and rules must be read with each fitted preprocessor's `get_feature_names_out()`. To go back to the 11 original variables, sum the columns of each nominal predictor (the names that start with `cat__proto_`, `cat__service_` or `cat__conn_state_`).

# Challenge B — storage

## Data

In [10]:
# Same names as in cyber, so that R5 can join the two blocks.
dev = pd.read_csv(STORAGE_DIR + "/storage_development.csv")
storage_groups = pd.read_csv(STORAGE_DIR + "/storage_groups.csv")
storage_folds = pd.read_csv(FOLDS_DIR + "/storage_folds.csv")

# raid is a nominal label, even though its levels contain numbers.
NOMINAL = ["load_type", "io_type", "raid", "device_type"]
NUMERIC = ["block_size", "n_jobs", "iodepth", "read_fraction", "n_disks"]
FEATURES = NOMINAL + NUMERIC

# The CSV has a different column order: compare the sorted names.
expected_columns = ["row_id"] + FEATURES + ["latency_ns"]
assert sorted(dev.columns) == sorted(expected_columns), "Unexpected storage schema"
assert "iops" not in dev.columns, "Excluded column iops must not be present"
X = dev[FEATURES].copy()
latency_ns = dev["latency_ns"].copy()

assert not dev.isna().any().any(), "Missing values in development data"
assert np.isfinite(X[NUMERIC].to_numpy()).all(), "Non-finite numeric predictors"
assert np.isfinite(latency_ns).all(), "Non-finite target values"
assert (latency_ns > 0).all(), "Latency must be strictly positive"
z = np.log10(latency_ns)

for column in ["row_id", "config_group", "feature_group", "fold_id", "latency_ns", "iops"]:
    assert column not in X.columns, "Forbidden predictor: " + column
assert X.shape[1] == 9, "Expected nine source predictors"

print("Development table:", dev.shape)
print("Predictors:", X.shape, "(", len(NOMINAL), "nominal and", len(NUMERIC), "numeric )")
print("Target: z = log10(latency_ns)")

Development table: (12190, 11)
Predictors: (12190, 9) ( 4 nominal and 5 numeric )
Target: z = log10(latency_ns)


In [11]:
# Check that the three tables are aligned before using positions.
assert list(storage_groups.columns) == ["row_id", "config_group"], "Unexpected group schema"
assert list(storage_folds.columns) == ["row_id", "fold_id"], "Unexpected fold schema"
assert dev["row_id"].is_unique, "Duplicate development row_id"
assert dev["row_id"].equals(storage_groups["row_id"]), "Group rows are not aligned"
assert dev["row_id"].equals(storage_folds["row_id"]), "Fold rows are not aligned"
assert sorted(storage_folds["fold_id"].unique()) == [0, 1, 2, 3, 4], "Expected five folds"

# Read the official assignment: the folds are never rebuilt.
folds = []
fold_rows = []
for fold_id in range(5):
    train_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() != fold_id)
    val_idx = np.flatnonzero(storage_folds["fold_id"].to_numpy() == fold_id)

    # The replicates of a configuration always stay together.
    train_groups = set(storage_groups.iloc[train_idx]["config_group"])
    val_groups = set(storage_groups.iloc[val_idx]["config_group"])
    assert train_groups.isdisjoint(val_groups), "Group leakage in fold " + str(fold_id)
    assert len(train_idx) + len(val_idx) == len(dev), "Incomplete partition"

    folds.append((train_idx, val_idx))
    fold_rows.append({"fold_id": fold_id, "training_rows": len(train_idx),
                      "validation_rows": len(val_idx)})

print("Configuration groups:", storage_groups["config_group"].nunique())
print(pd.DataFrame(fold_rows).to_string(index=False))
print("All five canonical folds passed the group separation checks.")

Configuration groups: 2438
 fold_id  training_rows  validation_rows
       0           9750             2440
       1           9750             2440
       2           9750             2440
       3           9755             2435
       4           9755             2435
All five canonical folds passed the group separation checks.


The storage development table has 12,190 rows and 9 source predictors (4 nominal and 5 numeric), grouped in 2,438 configurations. The five canonical folds are read from `../folds/storage_folds.csv`, and the checks above confirm that the replicates of a configuration never fall on both sides of a split. Identifiers, grouping metadata, fold assignments and the target are kept outside `X`; `iops` is absent.

The target is `z = log10(latency_ns)`, as fixed in P2.3. The original `latency_ns` is kept for physical-scale evaluation. R3 must invert its predictions with `10 ** z_hat` before computing any error in nanoseconds.

## Representation decisions (statement, Section 2)

**Scaling.** No scaler. The reasoning is the same as in cyber: a tree tests `x <= t` on one column, and an increasing transformation moves the threshold without changing which rows go to each side. The five numeric predictors (`block_size`, `n_jobs`, `iodepth`, `read_fraction`, `n_disks`) pass through in their original units, so a threshold on `n_disks` is read directly as a number of disks. P3 standardised them; P4 does not. The target transformation `z = log10(latency_ns)` is separate from this preprocessor and does not change.

**Categorical encoding.** `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` with every level kept. P3 removed `random`, `write`, `4+1` and `nvme`; a tree needs no reference level, and keeping them lets it ask about any single level, such as `device_type_hdd`. `raid` remains a nominal label: its levels are schemes, not ordered numbers.

**Unseen levels.** With `handle_unknown="ignore"`, an unseen level would be encoded as zeros for its predictor, meaning "none of the known levels". The checks below show that no validation row has an unseen level in any fold.

### Correspondence with P3

In [12]:
# Levels of each nominal predictor in the development table, and the P3 mapping.
P3_DROPPED_LEVELS = ["random", "write", "4+1", "nvme"]

mapping_rows = []
for column, dropped in zip(NOMINAL, P3_DROPPED_LEVELS):
    levels = sorted(X[column].unique())
    assert dropped in levels, "P3 reference level not found: " + dropped
    mapping_rows.append({"predictor": column, "levels": len(levels),
                         "P3_dropped_level": dropped,
                         "P3_indicator_columns": len(levels) - 1,
                         "P4_indicator_columns": len(levels)})

mapping = pd.DataFrame(mapping_rows)
print(mapping.to_string(index=False))
print("Total indicator columns, P3:", mapping["P3_indicator_columns"].sum())
print("Total indicator columns, P4:", mapping["P4_indicator_columns"].sum())

  predictor  levels P3_dropped_level  P3_indicator_columns  P4_indicator_columns
  load_type       2           random                     1                     2
    io_type       2            write                     1                     2
       raid       6              4+1                     5                     6
device_type       2             nvme                     1                     2
Total indicator columns, P3: 8
Total indicator columns, P4: 12


P3 dropped `load_type=random`, `io_type=write`, `raid=4+1` and `device_type=nvme`. P4 keeps them as columns, and it does not scale the numeric predictors. The four predictors have 2, 2, 6 and 2 levels, so the indicator columns go from 8 in P3 to 12 in P4. The check below confirms, on every fold, that each P3 indicator column has the same values as the P4 column of the same level.

## Preprocessing block (publish for R3 and R4)

In [13]:
def make_preprocessor():
    # One-hot with every level kept; unseen levels become all zeros.
    nominal_steps = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

    # Numeric predictors pass through unchanged: no scaler.
    preprocessor = ColumnTransformer([
        ("cat", nominal_steps, NOMINAL),
        ("num", "passthrough", NUMERIC),
    ])
    # Returned unfitted so that R3 puts it inside its Pipeline.
    return preprocessor

In [14]:
# Check the block inside each canonical fold: fitted on training rows only.
check_rows = []
for fold_id, (train_idx, val_idx) in enumerate(folds):
    preprocessor = make_preprocessor()
    assert not hasattr(preprocessor, "transformers_"), "Expected an unfitted transformer"
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]
    X_train_t = preprocessor.fit_transform(X_train)
    X_valid_t = preprocessor.transform(X_valid)

    encoder = preprocessor.named_transformers_["cat"]
    assert encoder.drop_idx_ is None, "No level may be dropped"

    # Count the columns and the validation levels that training never saw.
    n_cat = 0
    unseen_per_row = np.zeros(len(X_valid), dtype=int)
    for column, categories in zip(NOMINAL, encoder.categories_):
        assert list(categories) == sorted(X_train[column].unique()), "Wrong vocabulary"
        n_cat = n_cat + len(categories)
        unseen_per_row = unseen_per_row + (~X_valid[column].isin(categories)).to_numpy()
    n_num = len(NUMERIC)

    feature_names = preprocessor.get_feature_names_out()
    assert X_train_t.shape[1] == len(feature_names), "Missing feature names"
    assert X_valid_t.shape[1] == len(feature_names), "Inconsistent columns"
    assert len(feature_names) == n_cat + n_num, "Unexpected number of columns"
    assert np.isfinite(X_train_t).all(), "Non-finite training values"
    assert np.isfinite(X_valid_t).all(), "Non-finite validation values"

    # Numeric columns keep their original values: nothing was scaled.
    assert np.array_equal(X_train_t[:, n_cat:], X_train[NUMERIC].to_numpy()), "Train values changed"
    assert np.array_equal(X_valid_t[:, n_cat:], X_valid[NUMERIC].to_numpy()), "Valid values changed"

    # One indicator per predictor is 1 in training; an unseen level gives 0.
    train_ones = X_train_t[:, :n_cat].sum(axis=1)
    valid_ones = X_valid_t[:, :n_cat].sum(axis=1)
    assert (train_ones == len(NOMINAL)).all(), "Each training row needs one level per predictor"
    assert (valid_ones == len(NOMINAL) - unseen_per_row).all(), "Unseen levels must give zeros"

    if fold_id == 0:
        names_fold_0 = list(feature_names)

    check_rows.append({"fold_id": fold_id, "output_features": len(feature_names),
                       "indicator_columns": n_cat, "numeric_columns": n_num,
                       "unseen_validation_rows": int((unseen_per_row > 0).sum())})

print(pd.DataFrame(check_rows).to_string(index=False))
print("Fold 0 output names:", names_fold_0)
print("Preprocessing checks passed on all five folds.")

 fold_id  output_features  indicator_columns  numeric_columns  unseen_validation_rows
       0               17                 12                5                       0
       1               17                 12                5                       0
       2               17                 12                5                       0
       3               17                 12                5                       0
       4               17                 12                5                       0
Fold 0 output names: ['cat__load_type_random', 'cat__load_type_sequential', 'cat__io_type_read', 'cat__io_type_write', 'cat__raid_1+1', 'cat__raid_2+1', 'cat__raid_2+2', 'cat__raid_4+1', 'cat__raid_4+2', 'cat__raid_8+2', 'cat__device_type_hdd', 'cat__device_type_nvme', 'num__block_size', 'num__n_jobs', 'num__iodepth', 'num__read_fraction', 'num__n_disks']
Preprocessing checks passed on all five folds.


In [15]:
# Every P3 indicator column must equal the P4 column of the same level.
for fold_id, (train_idx, val_idx) in enumerate(folds):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[val_idx]

    p4 = make_preprocessor()
    p4.fit(X_train)
    p4_names = list(p4.named_transformers_["cat"].get_feature_names_out(NOMINAL))
    p4_valid = p4.named_transformers_["cat"].transform(X_valid[NOMINAL])

    p3 = OneHotEncoder(drop=P3_DROPPED_LEVELS, handle_unknown="ignore", sparse_output=False)
    p3.fit(X_train[NOMINAL])
    p3_names = list(p3.get_feature_names_out(NOMINAL))
    p3_valid = p3.transform(X_valid[NOMINAL])

    for position, name in enumerate(p3_names):
        assert name in p4_names, "P3 column missing in P4: " + name
        same_column = p4_valid[:, p4_names.index(name)]
        assert np.array_equal(p3_valid[:, position], same_column), "Different values: " + name
    print("Fold", fold_id, ": P3 columns", len(p3_names), "are all in P4 columns", len(p4_names))

Fold 0 : P3 columns 8 are all in P4 columns 12
Fold 1 : P3 columns 8 are all in P4 columns 12
Fold 2 : P3 columns 8 are all in P4 columns 12
Fold 3 : P3 columns 8 are all in P4 columns 12
Fold 4 : P3 columns 8 are all in P4 columns 12


In [16]:
# The target representation is unchanged from P2.3 and P3.
assert np.isfinite(z).all(), "Non-finite log target"
assert np.allclose(10 ** z, latency_ns), "Target inversion does not recover latency"
print("Target check passed: 10 ** z recovers latency_ns.")

Target check passed: 10 ** z recovers latency_ns.


**Checks.** All checks pass on the five canonical folds. The transformed matrix has 17 columns in every fold (12 indicators and 5 numeric), training and validation have the same columns, the numeric columns keep their original values, and no validation row has a level that training never saw. The target check confirms that `10 ** z` recovers `latency_ns`. These are preprocessing checks, not model results.

`make_preprocessor()` returns a new, unfitted transformer. R3 must place it inside the model `Pipeline`, fitted only on each fold's training rows. The output names (`cat__device_type_hdd`, `num__block_size`) can be used to read rules: a split on `cat__device_type_hdd` separates the hdd devices from the others.

## `features.md`

The dated Practice 4 entry (2026-11-17) is recorded in `../plantillas/features.md`. It preserves the P2 and P3 entries and documents the change of representation for the tree: no scaling and full one-hot encoding, with an unchanged predictive-information basis and route `primary`.